# Word-Level Dialect Analysis
Cross-compares every source word against every word in the DAT and DIT transcripts using:

- **Levenshtein similarity**
- **Position score** (normalised gap between source and target word index)

Scoring methods:
- **Combined Weighted Score**: $\alpha \cdot \text{lev\_sim} + (1-\alpha) \cdot \text{pos\_sim}$, $\alpha = 0.7$
- **Combined Harmonic Score**: $\frac{2 \cdot \text{lev\_sim} \cdot \text{pos\_sim}}{\text{lev\_sim} + \text{pos\_sim}}$

In [43]:
import sys
import importlib
import pandas as pd
from pathlib import Path

# Make scripts/ importable when running from inside scripts/
sys.path.insert(0, str(Path("__file__").resolve().parent))

import util.utils as utils
importlib.reload(utils)

from util.utils import (
    clean,
    build_word_comparison_df,
)

In [44]:
PROJECT_ROOT = Path("__file__").resolve().parent.parent
ANALYSIS_DIR = PROJECT_ROOT / "02-analysis-position"
DAT_TSV = ANALYSIS_DIR / "dialect-aware-transcript.tsv"
DIT_TSV = ANALYSIS_DIR / "dialect-ignorant-transcript.tsv"

## Load & merge data

In [45]:
df_dit = pd.read_csv(DIT_TSV, sep='\t', encoding='utf-8-sig')
df_dat = pd.read_csv(DAT_TSV, sep='\t', encoding='utf-8-sig')
df = pd.merge(df_dit, df_dat[['path', 'DAT']], on='path', how='inner')

print(f"Clips: {len(df)}")
df.head(3)

Clips: 100


,path,duration,sentence,sentence_source,client_id,dialect_region,canton,zipcode,age,gender,DIT,DAT
0,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,4.778662,Insgesamt habe ich einige Hunderttausend Frank...,news,300bb931-79ae-40ec-b989-3efd5e83f4c2,Zürich,ZH,8408.0,fourties,male,Insgesamt habe ich einige hundert Dusik vor un...,Insgesamt habe ich einige 100.000 Franken verl...
1,clips\6e084270-8d26-43d9-ba69-5e8ee793ab8c\dca...,5.294150,Welche Rolle hatte Rosenberg während des Holoc...,news,6e084270-8d26-43d9-ba69-5e8ee793ab8c,Zürich,ZG,6340.0,twenties,female,"Weli Rolle, höchter Uz stoodhergewerden von Ho...",Wer ironne hättet viele bands auf den einzugem...
2,clips\6858a37b-edd0-4fdf-871c-96d3b1bd3e21\82f...,5.802676,Das ist angesichts aller schlechten Optionen d...,news,6858a37b-edd0-4fdf-871c-96d3b1bd3e21,Zürich,ZH,8704.0,fourties,male,Das ist angesichts vor allen Schlachten Option...,Das ist angesichts von allen schlechten Option...


## Determine global normalisation constant

In [46]:
all_words = (
    df['sentence'].dropna().str.split().explode().tolist()
    + df['DIT'].dropna().str.split().explode().tolist()
    + df['DAT'].dropna().str.split().explode().tolist()
)

all_sentences = pd.concat([
    df['sentence'].dropna(),
    df['DIT'].dropna(),
    df['DAT'].dropna()
])

global_max_word_length     = max(len(clean(w)) for w in all_words)
global_max_sentence_length = max(len(str(s).split()) for s in all_sentences)

print(f"Global max word length     (Levenshtein normaliser): {global_max_word_length}")
print(f"Global max sentence length (position normaliser):    {global_max_sentence_length}")

Global max word length     (Levenshtein normaliser): 27
Global max sentence length (position normaliser):    65


## Cross-comparison: every source word × every target position

In [47]:
# Global normalisation: divide by longest word across the entire dataset
word_df = build_word_comparison_df(df, global_max_word_length, global_max_sentence_length)
print(f"Total rows: {len(word_df):,}")
word_df.head(10)

Total rows: 9,441


,clip_id,src_word_index,target_word_index,is_same_position,src_word,dit_word,dat_word,dit_word_score,dat_word_score,position_score,dit_combined_weighted,dit_combined_harmonic,dat_combined_weighted,dat_combined_harmonic,src_len,dit_len,dat_len
0,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,0,1,insgesamt,insgesamt,insgesamt,1.000,1.000,1.000,1.000,1.000,1.000,1.000,7,9,7
1,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,1,0,insgesamt,habe,habe,0.704,0.704,0.984,0.788,0.821,0.788,0.821,7,9,7
2,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,2,0,insgesamt,ich,ich,0.704,0.704,0.969,0.783,0.816,0.783,0.816,7,9,7
3,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,3,0,insgesamt,einige,einige,0.778,0.778,0.953,0.831,0.857,0.831,0.857,7,9,7
4,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,4,0,insgesamt,hundert,100000,0.741,0.667,0.938,0.800,0.828,0.748,0.780,7,9,7
5,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,5,0,insgesamt,dusik,franken,0.704,0.667,0.922,0.769,0.798,0.744,0.774,7,9,7
6,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,6,0,insgesamt,vor,verloren,0.667,0.667,0.906,0.739,0.768,0.739,0.768,7,9,7
7,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,7,0,insgesamt,und,N/A,0.704,0.667,0.891,0.760,0.787,0.734,0.763,7,9,7
8,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,8,0,insgesamt,verloren,N/A,0.667,0.667,0.875,0.729,0.757,0.729,0.757,7,9,7
9,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,1,0,0,habe,insgesamt,insgesamt,0.704,0.704,0.984,0.788,0.821,0.788,0.821,7,9,7


## Summary statistics

In [48]:
word_df[['dit_word_score', 'dat_word_score', 'position_score',
         'dit_combined_weighted', 'dat_combined_weighted',
         'dit_combined_harmonic', 'dat_combined_harmonic'
         ]].describe().round(3)

,dit_word_score,dat_word_score,position_score,dit_combined_weighted,dat_combined_weighted,dit_combined_harmonic,dat_combined_harmonic
count,9441.000,9441.000,9441.000,9441.000,9441.000,9441.000,9441.000
mean,0.784,0.787,0.897,0.818,0.820,0.823,0.824
std,0.116,0.120,0.156,0.094,0.095,0.127,0.127
min,0.074,0.185,0.000,0.324,0.369,0.000,0.000
25%,0.741,0.741,0.906,0.767,0.767,0.790,0.789
50%,0.815,0.815,0.953,0.838,0.835,0.857,0.857
75%,0.852,0.889,0.969,0.885,0.887,0.898,0.900
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000
